# Lab: Patent Phrase Matching with a Transformer Cross-Encoder

**Name:** Avval Manjhu **Roll no. A24ARIU0025  **Date:** 1 nov.

## Overview

In this lab you fine-tune a pretrained Transformer **cross-encoder** (DeBERTa-v3-small) to predict how similar two patent phrases are, judged within a patent context. You will compare it against simple baselines, run controlled experiments, analyse its errors, and produce a Kaggle `submission.csv`.

The data comes from the Kaggle competition **U.S. Patent Phrase to Phrase Matching**. Each row has an **anchor**, a **target**, a **context** (a CPC code such as `G02`) and a **score** in {0, 0.25, 0.5, 0.75, 1}. The metric is the **Pearson correlation** between predictions and true scores.

This single notebook contains the whole lab: weight download, the working pipeline, baselines, experiments, error analysis, submission, and written answers.

## Learning objectives

1. Frame text-pair similarity as regression and fine-tune a cross-encoder with Hugging Face `Trainer`.
2. Build a leak-free validation split (grouped by anchor) and use it for fair comparisons.
3. Compare a fine-tuned model against non-neural and frozen-embedding baselines.
4. Run controlled experiments (change one setting at a time) and report them honestly.
5. Analyse where and why the model fails.

## Tasks and deliverables

| # | Task | Where |
|---|------|-------|
| 1 | Run the pipeline end to end and obtain a validation Pearson | Sections 1-9 |
| 2 | Baselines without fine-tuning, on the same validation split | Section 7 |
| 3 | At least **two** controlled experiments, one setting changed each | Section 10 |
| 4 | Error analysis using the validation predictions | Section 11 |
| 5 | Produce `submission.csv`, make a **late submission**, record public and private scores | Sections 12-13 |
| 6 | Written reflection | Section 14 |

## How to run

- Kaggle notebook, **GPU T4 or P100**. Add the competition data via **Add Input → Competitions → U.S. Patent Phrase to Phrase Matching**.
- The competition scores notebooks with **internet off**, so the pretrained checkpoint must be attached as an input. Section 2 explains how to create it (first run only, internet on) and the notebook finds it automatically afterwards.
- Writes to `/kaggle/working/`: `submission.csv`, `val_predictions.csv`, `experiment_results.csv`. A rerun overwrites them. No model checkpoints are saved.
- Uses packages preinstalled on Kaggle (`torch`, `transformers`, `datasets`, `scikit-learn`, `pandas`, `huggingface_hub`).

## 1. Setup

In [1]:
import glob
import gc
import os
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
import transformers
from datasets import Dataset
from IPython.display import display
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModel, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)

torch 2.11.0+cu130 | transformers 5.21.0 | device: cuda


## 2. Configuration and pretrained weights

These are the settings you will vary in your experiments. Change **one** setting at a time so you can tell what caused a difference.

**Offline weights (important).** The competition scores your notebook with internet off, so `from_pretrained("microsoft/deberta-v3-small")` cannot download anything at submission time. The checkpoint must be attached as an input:

1. Create a **separate new Kaggle notebook**, turn **Internet on** (Settings → Internet), no GPU needed.
2. Paste the cell `download_checkpoint` below (it is self-contained), set `MODEL_ID`, and run **Save Version → Save & Run All (Commit)**.
3. In this lab notebook: **Add Input → Your Work → Notebooks**, then select that notebook. Its output appears under `/kaggle/input/` and `get_model_path` finds it automatically.

If no local copy is found, `get_model_path` tries to download it into `/kaggle/working/`, which only works with internet on. Check each checkpoint's model card for its license.

In [7]:
import kagglehub
# You may need to re-run this cell after logging in.
kagglehub.login()
path = kagglehub.competition_download('us-patent-phrase-to-phrase-matching')

100%|██████████| 682k/682k [00:00<00:00, 104MB/s]

Extracting files...


In [8]:
MODEL_ID = "microsoft/deberta-v3-small"   # Hub name; the local copy is found by its last path component
MAX_LEN = 64            # anchor + target + context fit easily within 64 tokens
EPOCHS = 3
LR = 2e-5
BATCH_SIZE = 32
VAL_FRACTION = 0.2      # share of *anchors* (not rows) held out for validation
CONTEXT_MODE = "section"   # "none" | "section" (CPC section name) | "code" (raw CPC code)
OUT_DIR = "/kaggle/working"
RUN_EXPERIMENTS = True     # set False for the final offline submission run to save time

In [9]:
# ---- self-contained helper: copy THIS cell alone into a separate internet-on notebook to create the offline weights ----
def download_checkpoint(model_id, out_root="/kaggle/working"):
    from huggingface_hub import snapshot_download
    save_dir = Path(out_root) / model_id.split("/")[-1]
    snapshot_download(
        repo_id=model_id,
        local_dir=save_dir,
        # PyTorch weights, configs, and tokenizer files only (skips TensorFlow/Flax/ONNX copies)
        allow_patterns=["*.json", "*.safetensors", "*.bin", "*.model", "*.txt"],
    )
    print(sorted(p.name for p in save_dir.iterdir()))
    return str(save_dir)

# In the helper notebook you would just run:  download_checkpoint(MODEL_ID)

In [12]:
def find_one(pattern):
    '''Return the directory of the first file under /kaggle/input that matches the pattern.'''
    hits = glob.glob(f"/kaggle/input/**/{pattern}", recursive=True)
    return os.path.dirname(hits[0]) if hits else None


def get_model_path(model_id):
    '''Prefer an attached local checkpoint; otherwise download (needs internet).'''
    name = model_id.split("/")[-1]
    local = find_one(f"{name}/config.json")
    if local:
        return local
    saved = Path(OUT_DIR) / name
    if not (saved / "config.json").exists():
        try:
            download_checkpoint(model_id, OUT_DIR)
        except Exception as e:
            raise RuntimeError(
                f"No local copy of {model_id} found under /kaggle/input and the download failed ({e}). "
                "Attach the helper notebook's output as an input, or turn Internet on.") from e
    return str(saved)


def find_data_dir(roots=("/kaggle/input", "/root/.cache/kagglehub")):
    for base in roots:
        for root, dirs, files in os.walk(base, followlinks=True):
            if "train.csv" in files and "test.csv" in files:
                return root
    return None


DATA_DIR = find_data_dir()
assert DATA_DIR, ("Competition data not found under /kaggle/input. Join the competition, use Add Input -> Competitions, "
                  "then restart the session and run from the first cell.")
MODEL_PATH = get_model_path(MODEL_ID)
print("data:", DATA_DIR)
print("model:", MODEL_PATH, "(local)" if MODEL_PATH != MODEL_ID else "")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

['.cache', 'config.json', 'pytorch_model.bin', 'spm.model', 'tokenizer_config.json']
data: /root/.cache/kagglehub/competitions/us-patent-phrase-to-phrase-matching
model: /kaggle/working/deberta-v3-small (local)


## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [13]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()

train: (36473, 5) | test (visible): (36, 4)
score
0.00     7471
0.25    11519
0.50    12300
0.75     4029
1.00     1154
Name: count, dtype: int64


,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00


## 4. Model input

The model reads the anchor as the first text segment, and the target plus a short context description as the second. The tokenizer joins them as `[CLS] anchor [SEP] target; context [SEP]`, so attention can compare the two phrases token by token: this is what makes it a **cross-encoder**.

How you represent context is one of the experiments you can run, so `build_inputs` supports three modes: `none` (target only), `section` (target + CPC section name, the default), and `code` (target + the raw CPC code).

In [14]:
CPC_SECTIONS = {
    "A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting",
    "G": "physics", "H": "electricity",
}


def build_inputs(df, context_mode="section"):
    df = df.copy()
    df["text_a"] = df["anchor"]
    if context_mode == "none":
        df["text_b"] = df["target"]
    elif context_mode == "section":
        df["text_b"] = df["target"] + "; " + df["context"].str[0].map(CPC_SECTIONS)
    elif context_mode == "code":
        df["text_b"] = df["target"] + "; " + df["context"]
    else:
        raise ValueError(f"unknown context_mode: {context_mode}")
    return df


build_inputs(train, CONTEXT_MODE)[["text_a", "text_b", "score"]].head()

,text_a,text_b,score
0,abatement,abatement of pollution; human necessities,0.50
1,abatement,act of abating; human necessities,0.75
2,abatement,active catalyst; human necessities,0.25
3,abatement,eliminating process; human necessities,0.50
4,abatement,forest region; human necessities,0.00


## 5. Validation split

The training set contains only about 730 distinct anchors, each paired with many targets. If the same anchor appeared in both the training and validation sets, the validation score would overestimate performance on unseen anchors. The split therefore holds out whole **anchors**.

Keep this split fixed (same `SEED` and `VAL_FRACTION`) for every experiment, so your comparisons are fair.

In [15]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

train rows 28914 | val rows 7559 | val anchors 147


## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

Some `transformers` 5.x versions build this tokenizer without `[CLS]`/`[SEP]`; `make_tokenizer` detects that and sets the pair template explicitly.

In [16]:
def make_tokenizer(path):
    tok = AutoTokenizer.from_pretrained(path)
    if tok("a", "b")["input_ids"][0] != tok.cls_token_id:   # special tokens missing: set the pair template
        tok.backend_tokenizer.post_processor = TemplateProcessing(
            single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
            special_tokens=[("[CLS]", tok.cls_token_id), ("[SEP]", tok.sep_token_id)])
    return tok


def to_dataset(df, tok, max_len, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    d = df[cols].astype({"score": "float32"}) if with_labels else df[cols]   # MSE loss needs float32 labels
    ds = Dataset.from_pandas(d)
    ds = ds.map(lambda b: tok(b["text_a"], b["text_b"], truncation=True, max_length=max_len),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


tokenizer = make_tokenizer(MODEL_PATH)
_demo = to_dataset(build_inputs(trn.head(3), CONTEXT_MODE), tokenizer, MAX_LEN)
print(tokenizer.decode(_demo[0]["input_ids"]))

[transformers] The tokenizer you are loading from '/kaggle/working/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/3 [00:00<?, ? examples/s]

[CLS] abatement[SEP] abatement of pollution; human necessities[SEP]


## 7. Baselines without fine-tuning (Task 2)

Before trusting a fine-tuned model, check what you get for free. Both baselines are scored on the **same validation split**:

- **Word overlap (Jaccard):** the share of shared lowercase words between anchor and target. No learning at all.
- **Frozen embeddings:** the pretrained model, *not* fine-tuned, embeds anchor and target separately (mean pooling); the score is their cosine similarity. This is a bi-encoder, so it cannot compare the phrases token by token.

In [17]:
def pearson(a, b):
    return float(np.corrcoef(np.asarray(a).reshape(-1), np.asarray(b).reshape(-1))[0, 1])


def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / max(len(sa | sb), 1)


@torch.no_grad()
def embed(texts, tok, mdl, bs=128):
    out = []
    for i in range(0, len(texts), bs):
        enc = tok(texts[i:i + bs], padding=True, truncation=True, max_length=MAX_LEN, return_tensors="pt").to(device)
        h = mdl(**enc).last_hidden_state
        m = enc["attention_mask"].unsqueeze(-1).float()
        out.append(((h * m).sum(1) / m.sum(1)).cpu())
    return torch.cat(out)


baseline_results = []

val_jac = [jaccard(a, t) for a, t in zip(val.anchor, val.target)]
baseline_results.append({"method": "Jaccard word overlap", "val_pearson": pearson(val_jac, val.score)})

frozen = AutoModel.from_pretrained(MODEL_PATH).float().to(device).eval()
ea, et = embed(val.anchor.tolist(), tokenizer, frozen), embed(val.target.tolist(), tokenizer, frozen)
val_cos = F.cosine_similarity(ea, et).numpy()
baseline_results.append({"method": "Frozen DeBERTa, cosine similarity", "val_pearson": pearson(val_cos, val.score)})
del frozen, ea, et
gc.collect(); torch.cuda.empty_cache()

pd.DataFrame(baseline_results)

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: /kaggle/working/deberta-v3-small
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,method,val_pearson
0,Jaccard word overlap,0.424901
1,"Frozen DeBERTa, cosine similarity",0.197904


## 8. Fine-tuning

With `num_labels=1`, the classification head outputs a single number. Because the labels are floats, Hugging Face treats the task as **regression** and minimizes mean squared error. The competition metric is the **Pearson correlation** between predictions and true scores. Pearson correlation ignores any linear rescaling of the predictions, so the model is rewarded for ranking pairs in the right order, not for hitting the exact values.

`run_training` wraps the whole train-and-validate loop so the main run and the experiments in Section 10 use identical code.

In [18]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": pearson(preds, labels)}


def load_model(path):
    # Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision needs
    # fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
    try:
        return AutoModelForSequenceClassification.from_pretrained(path, num_labels=1, dtype=torch.float32)
    except TypeError:
        return AutoModelForSequenceClassification.from_pretrained(path, num_labels=1, torch_dtype=torch.float32)


def make_args(**kw):
    # transformers >= 4.41 calls it eval_strategy; older versions call it evaluation_strategy
    try:
        return TrainingArguments(eval_strategy="epoch", **kw)
    except TypeError:
        return TrainingArguments(evaluation_strategy="epoch", **kw)


def run_training(model_path=MODEL_PATH, context_mode=CONTEXT_MODE, max_len=MAX_LEN,
                 epochs=EPOCHS, lr=LR, batch_size=BATCH_SIZE):
    transformers.set_seed(SEED)
    tok = make_tokenizer(model_path)
    ds_trn = to_dataset(build_inputs(trn, context_mode), tok, max_len)
    ds_val = to_dataset(build_inputs(val, context_mode), tok, max_len)
    model = load_model(model_path)
    steps_per_epoch = -(-len(ds_trn) // batch_size)         # ceiling division
    warmup_steps = int(0.1 * steps_per_epoch * epochs)      # 10% of all training steps (warmup_ratio was removed in transformers 5)
    args = make_args(
        output_dir=f"{OUT_DIR}/trainer_tmp",
        num_train_epochs=epochs,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=2 * batch_size,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        report_to="none",
        seed=SEED,
    )
    trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                      data_collator=DataCollatorWithPadding(tok), compute_metrics=compute_metrics)
    trainer.train()
    val_pred = trainer.predict(ds_val).predictions.reshape(-1)
    return {"trainer": trainer, "tok": tok, "val_pred": val_pred, "val_pearson": pearson(val_pred, val.score),
            "cfg": dict(context_mode=context_mode, max_len=max_len, epochs=epochs, lr=lr, batch_size=batch_size)}


main = run_training()
print(f"Main run validation Pearson: {main['val_pearson']:.4f}")

[transformers] The tokenizer you are loading from '/kaggle/working/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/working/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
poo

Epoch,Training Loss,Validation Loss,Pearson
1,0.029541,0.028131,0.803093
2,0.022504,0.024395,0.819427
3,0.019434,0.024358,0.820304


Main run validation Pearson: 0.8203


## 9. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [19]:
val["pred"] = main["val_pred"]
print(f"Validation Pearson: {pearson(val['pred'], val['score']):.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

summary = pd.DataFrame(baseline_results + [{"method": "Fine-tuned DeBERTa cross-encoder", "val_pearson": main["val_pearson"]}])
summary

Validation Pearson: 0.8203


,method,val_pearson
0,Jaccard word overlap,0.424901
1,"Frozen DeBERTa, cosine similarity",0.197904
2,Fine-tuned DeBERTa cross-encoder,0.820304


## 10. Controlled experiments (Task 3)

Run **at least two** experiments, each changing **exactly one** setting from the main run (Section 2 or Section 4). The three below are provided as defaults; edit the list to try your own ideas, for example `max_len`, `batch_size`, or another checkpoint via `model_path` (attach its weights first).

Each experiment retrains from scratch on the same split and seed, so differences are attributable to the one change. A single run per setting is noisy; treat differences of a few thousandths as inconclusive.

In [20]:
EXPERIMENTS = [
    {"name": "context = none (target only)", "context_mode": "none"},
    {"name": "context = raw CPC code",       "context_mode": "code"},
    {"name": "lr = 5e-5",                    "lr": 5e-5},
]

rows = [{"experiment": "main run (reference)", "changed": "-", "val_pearson": main["val_pearson"]}]
if RUN_EXPERIMENTS:
    for exp in EXPERIMENTS:
        changes = {k: v for k, v in exp.items() if k != "name"}
        print(f"\n=== {exp['name']} ===")
        r = run_training(**changes)
        rows.append({"experiment": exp["name"], "changed": str(changes), "val_pearson": r["val_pearson"]})
        del r
        gc.collect(); torch.cuda.empty_cache()

exp_results = pd.DataFrame(rows)
exp_results["delta_vs_main"] = exp_results["val_pearson"] - main["val_pearson"]
exp_results.to_csv(f"{OUT_DIR}/experiment_results.csv", index=False)
exp_results


=== context = none (target only) ===


[transformers] The tokenizer you are loading from '/kaggle/working/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

...

Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/working/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
poo

Epoch,Training Loss,Validation Loss,Pearson
1,0.027534,0.024399,0.808354
2,0.018984,0.023586,0.820211
3,0.014179,0.023262,0.823389


,experiment,changed,val_pearson,delta_vs_main
0,main run (reference),-,0.820304,0.000000
1,context = none (target only),{'context_mode': 'none'},0.816700,-0.003604
2,context = raw CPC code,{'context_mode': 'code'},0.819187,-0.001116
3,lr = 5e-5,{'lr': 5e-05},0.823389,0.003085


## 11. Error analysis (Task 4)




In [21]:
va = pd.read_csv(f"{OUT_DIR}/val_predictions.csv")
va["abs_err"] = (va["pred"].clip(0, 1) - va["score"]).abs()
va["section"] = va["context"].str[0]

print("Worst 15 predictions:")
display(va.sort_values("abs_err", ascending=False).head(15)[["anchor", "target", "context", "score", "pred"]].round(3))

print("\nError by true score:")
display(va.groupby("score")["abs_err"].agg(["mean", "count"]).round(3))

print("\nError and Pearson by CPC section:")
rows_sec = []
for sec, g in va.groupby("section"):
    rows_sec.append({"section": sec, "n": len(g), "mean_abs_err": g["abs_err"].mean(),
                     "pearson": g["pred"].corr(g["score"]) if len(g) > 10 else np.nan})
display(pd.DataFrame(rows_sec).set_index("section").round(3))

Worst 15 predictions:


,anchor,target,context,score,pred
5836,punch face,face punch,B05,0.00,0.920
972,boom hydraulic cylinder,boom a hydraulic cylinder,E02,0.00,0.830
3794,imaging axis,imaging of axis,G02,0.00,0.820
3790,imaging axis,image of axis,G02,0.00,0.766
2776,end pins,wood pin,G10,0.75,-0.008
1154,breech mechanism,breech loading naval gun,B29,0.75,0.006
490,average pore size,average cell size,B24,0.75,0.031
6005,return structure,return to structure,D01,0.00,0.697
7499,whose identity,customer id,G06,0.75,0.065
6259,running tally,credit total,G08,0.75,0.097



Error by true score:


,mean,count
score,,
0.00,0.125,1522
0.25,0.097,2465
0.50,0.107,2475
0.75,0.144,832
1.00,0.042,265



Error and Pearson by CPC section:


,n,mean_abs_err,pearson
section,,,
A,943,0.116,0.786
B,1926,0.086,0.894
C,1177,0.119,0.770
D,199,0.092,0.789
E,373,0.113,0.809
F,968,0.101,0.848
G,1040,0.133,0.706
H,933,0.120,0.805


In [22]:
# Over- vs under-prediction: signed error by true score
va["signed_err"] = va["pred"].clip(0, 1) - va["score"]
print(va.groupby("score")["signed_err"].mean().round(3))

# Do the worst errors cluster on particular anchors?
print(va.groupby("anchor")["abs_err"].mean().sort_values(ascending=False).head(10).round(3))

score
0.00    0.125
0.25    0.055
0.50   -0.008
0.75   -0.089
1.00   -0.042
Name: signed_err, dtype: float64
anchor
running tally          0.226
killed                 0.212
whose identity         0.209
secure to guard        0.199
vegfr2                 0.193
fdi                    0.182
cochineal              0.179
key recesses           0.176
antiatherosclerotic    0.174
display object         0.168
Name: abs_err, dtype: float64


## 12. Submission file (Task 5)

In [23]:
trainer, tok, cfg = main["trainer"], main["tok"], main["cfg"]
ds_test = to_dataset(build_inputs(test, cfg["context_mode"]), tok, cfg["max_len"], with_labels=False)
test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

,id,score
0,4112d61851461f60,0.566895
1,09e418c93a776564,0.787598
2,36baf228038e314b,0.502441
3,1f37ead645e7f0c8,0.327637
4,71a5b6ad068d531f,0.000000
